In [1]:
!pip install -q unsloth
!pip install -q transformers datasets trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "/content/drive/MyDrive/Task2_Hindi/dataset_clean.jsonl",
    split = "train"
)

print(f"Total examples: {len(dataset)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Generating train split: 0 examples [00:00, ? examples/s]

Total examples: 1221


In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "sarvamai/sarvam-1",
    max_seq_length = 512,
    dtype = torch.float16,
    load_in_4bit = True,
)

print("Model loaded successfully")
print(f"Parameters: {model.num_parameters():,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Unsloth: Will load sarvamai/sarvam-1 as a legacy tokenizer.


sarvamai/sarvam-1 does not have a padding token! Will use pad_token = <unk>.
Model loaded successfully
Parameters: 2,525,087,744


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 23,969,792 || all params: 2,549,057,536 || trainable%: 0.9403


In [5]:
def format_example(example):
    messages = example["messages"]
    system    = messages[0]["content"]
    user      = messages[1]["content"]
    assistant = messages[2]["content"]
    text = (
        f"### System:\n{system}\n\n"
        f"### User:\n{user}\n\n"
        f"### Assistant:\n{assistant}"
    )
    return {"text": text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

split1 = dataset.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split1["train"]
eval_dataset  = split2["train"]
test_dataset  = split2["test"]

print(f"Train      : {len(train_dataset)}")
print(f"Validation : {len(eval_dataset)}")
print(f"Test       : {len(test_dataset)}")
print(f"\nSample:")
print(train_dataset[0]["text"][:300])

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

Train      : 976
Validation : 122
Test       : 123

Sample:
### System:
तुम एक सहायक हिन्दी भाषा के सहायक हो।

### User:
सब्जी का सूप कैसे बनाएँ?

### Assistant:
सब्जी का सूप बनाने के लिए गाजर, टमाटर, प्याज और मटर छोटे टुकड़ों में काटें। कड़ाही में मक्खन गरम करें और प्याज भूनें। बाकी सब्जियाँ डालकर पाँच मिनट भूनें। तीन कप पानी, नमक, काली मिर्च और हर्ब्स डाले


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    output_dir = "outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    num_train_epochs = 3,
    learning_rate = 1e-4,
    lr_scheduler_type = "cosine",
    warmup_steps = 20,
    weight_decay = 0.01,
    max_grad_norm = 0.3,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "no",
    optim = "adamw_8bit",
    seed = 42,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = training_args,
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete")
print(f"Total steps : {trainer_stats.global_step}")
print(f"Final loss  : {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/976 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/122 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 976 | Num Epochs = 3 | Total steps = 183
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 23,969,792 of 2,549,057,536 (0.94% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.979488,0.954401
100,0.854691,0.885798
150,0.758735,0.869987
183,0.745375,0.868024


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


Training complete
Total steps : 183
Final loss  : 0.9745


In [7]:
save_path = "/content/drive/MyDrive/Task2_Hindi/sarvam_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved to {save_path}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Task2_Hindi/sarvam_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Task2_Hindi/sarvam_adapter.


Saved to /content/drive/MyDrive/Task2_Hindi/sarvam_adapter


In [8]:
FastLanguageModel.for_inference(model)

def generate(prompt, max_new_tokens=256):
    text = (
        f"### System:\nतुम एक सहायक हिन्दी भाषा के सहायक हो।\n\n"
        f"### User:\n{prompt}\n\n"
        f"### Assistant:\n"
    )
    inputs = tokenizer(
        text,
        return_tensors = "pt",
        truncation = True,
        max_length = 512
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = max_new_tokens,
        temperature = 0.3,
        top_p = 0.85,
        top_k = 50,
        repetition_penalty = 1.15,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][len(inputs["input_ids"][0]):],
        skip_special_tokens = True
    ).strip()
    return response

test_prompts = [
    "भारत का सबसे बड़ा रेगिस्तान कौन सा है?",
    "दोस्त से झगड़ा हो जाए तो क्या करें?",
    "एक किसान के पास 120 आम हैं। वह उन्हें 8 टोकरियों में बराबर बाँटता है। हर टोकरी में कितने आम होंगे?",
    "सुबह जल्दी उठने की आदत कैसे डालें?",
    "नमस्ते कहने के अलावा हिंदी में अभिवादन के और क्या तरीके हैं?"
]

print("=" * 60)
for i, prompt in enumerate(test_prompts):
    print(f"\nप्रश्न {i+1}: {prompt}")
    print("-" * 40)
    print(f"उत्तर: {generate(prompt)}")
    print("=" * 60)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



प्रश्न 1: भारत का सबसे बड़ा रेगिस्तान कौन सा है?
----------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


उत्तर: भारत का सबसे बड़ा रेगिस्तान थार ఎడారి है, जो राजस्थान राज्य में स्थित है और लगभग 20 लाख वर्ग किलोमीटर क्षेत्रफल में फैला हुआ है। यह अत्यधिक शुष्क जलवायु के लिए जाना जाता है जहां वर्षा बहुत कम होती है। यहाँ की रेत बेहद कठोर होती है जिससे खेती करना मुश्किल होता है लेकिन यहां बकरियाँ पालने और खनिज संसाधनों से समृद्धता पाई जाती है। इस रेगिस्तान को भारत का महान भारतीय रेगिस्तान भी कहा जाता है क्योंकि इसकी विशालता और महत्व इसे देश के रेगिस्तानी क्षेत्रों में अलग बनाता है। यह रेगिस्तान अपनी भौगोलिक विशेषताओं के कारण कई प्रकार के पौधों और जानवरों का घर भी है। इसलिए यह पर्यावरण संतुलन बनाए रखने में महत्वपूर्ण भूमिका निभाता है। थार रेगिस्तान अपने पारिस्थितिक महत्व के साथ ही सांस्कृतिक और ऐतिहासिक दृष्टि से भी अत्यंत महत्वपूर्ण स्थान रखता है। यह भारत के रेगिस्तानी क्षेत्रों की विविधता और चुनौतियों का प्रतीक है जिसके अध्ययन और संरक्षण की आवश्यकता बढ़ती जा रही है ताकि भविष्य की पीढ़ियां इसका लाभ उठा सकें। </sतुम एक सहायक हिन्दी भाषा के सहायक हो।

प्रश्न 2: दोस्त से झगड़ा हो जाए तो क्या करें?

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


उत्तर: जब दोस्ती में मतभेद हों, तो शांत और समझदारी से बात करना जरूरी होता है ताकि स्थिति बिगड़ने न पाए। सबसे पहले दोनों पक्षों को अपनी बात रखने का मौका दें। यदि आप भी अपनी राय रखें लेकिन सम्मानजनक तरीके से, तो बातचीत बेहतर होती है। अगर फिर भी समस्या दूर नहीं हुई, तो थोड़ा समय निकालना फायदेमंद हो सकता है। कभी-कभी कुछ दिनों बाद सामने वाले की सोच बदल जाती है या वह माफ़ी मांगता है जिससे संबंध सुधारने में मदद मिलती है। हमेशा याद रखना कि हर रिश्ता नाजुक होता है और उसे धीरे-धीरे मजबूत करने की आवश्यकता होती है। छोटी सी असहमति बड़ी बहस में बदलने देने से रिश्ते पर बुरा प्रभाव पड़ता है इसलिए संवाद बनाए रखना बहुत जरूरी होता है। इस तरह की स्थितियों में धैर्य और समझदारी आवश्यक होती है जो रिश्तों को स्वस्थ बनाने में मदद करती हैं। जब विवाद होते हैं तब तुरंत प्रतिक्रिया देना आसान लगता है लेकिन संयम से काम लेने से संबंधों को नुकसान कम होता है। यह व्यवहार भविष्य में भी सकारात्मक परिणाम देता है क्योंकि इससे विश्वास बना रहता है और संबंध अधिक मजबूत बनते हैं। इसलिए सभी रिश्तों में संचार और सहिष्णुता महत्वपूर

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


उत्तर: टोकरी में कुल आम = 120 = 4 गुना 30 = 60 आम। इसलिए प्रत्येक टोकरी में 60/8 = 7.5 आम होंगे, लेकिन चूंकि यह पूर्ण संख्या नहीं होती, इसलिए हम इसे निकटतम पूर्णांक तक घटाते हैं और 7 आम प्रति टोकरी प्राप्त करते हैं। इस प्रकार का प्रश्न भिन्नताओं को समझने में मदद करता है और रोजमर्रा की स्थितियों में उपयोगी होता है। यह गणित आधारित प्रश्नों में महत्वपूर्ण अवधारणाओं को दर्शाता है जो वास्तविक जीवन में भी लागू होते हैं। इस तरह के प्रश्नोत्तर से तर्क कौशल मजबूत होता है और समस्या समाधान क्षमता बढ़ती है। इस प्रकार के प्रश्नों पर ध्यान देना आवश्यक है क्योंकि वे न केवल गणना बल्कि निर्णय लेने की क्षमता भी बढ़ाते हैं। इस प्रकार के प्रश्नों का अभ्यास करने से व्यक्ति अधिक व्यवस्थित तरीके से समस्याओं को हल करना सीख सकता है जिससे उसकी सोच तेज और सटीक बनती है। इसलिए ऐसे प्रश्नों को नियमित रूप से पढ़ने और अभ्यास करने से विश्लेषणात्मक सोच बेहतर होती है। इस प्रकार के प्रश्नों को गंभीरता से समझना जरूरी होता है ताकि सही उत्तर निकाला जा सके। इसलिए ये प्रश्न शिक्षा प्रणाली में बहुत महत्वपूर्ण माने जाते हैं। इन

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


उत्तर: सुबह जल्दी उठना बहुत जरूरी है क्योंकि इससे दिनचर्या व्यवस्थित रहती है और उत्पादकता बढ़ती है। सबसे पहले अलार्म लगाएँ जो आपको समय पर जगाए रखेगा। सुबह में चाय या कॉफी पीते हुए व्यायाम करें ताकि शरीर सक्रिय रहे। सोने से कुछ घंटे पहले स्क्रीन बंद कर दें जिससे नींद अच्छी आती है। सुबह धूप में निकलें ताकि विटामिन डी प्राप्त हो सके। नियमित अभ्यास करने से यह आदत बन जाती है जिसे बदलना मुश्किल होता है इसलिए धीरे-धीरे शुरुआत करना बेहतर होता है। इस तरह आप स्वस्थ जीवनशैली अपना सकते हैं। जागने का सही समय निर्धारित करके उसे बनाए रखना महत्वपूर्ण है। इसके लिए अनुशासन आवश्यक है लेकिन साथ ही मन को सकारात्मकता से भरकर इसे आसान बनाया जा सकता है। सुबह जल्दी उठना स्वास्थ्य, मानसिक संतुलन और कार्य क्षमता बढ़ाने वाला होता है। यदि कोई व्यक्ति इस आदत को अपना लेता है तो वह अधिक सफल और खुशहाल जीवन जी सकता है। इसलिए हर किसी को अपनी दिनचर्या में बदलाव लाने चाहिए। सुबह जल्दी उठना व्यक्तिगत विकास और सफलता का संकेत माना जाता है। इसका पालन करने वाले लोग अपने लक्ष्यों को आसानी से पूरा करते हैं और दूसरों के सामने अच्

In [10]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

from unsloth import FastLanguageModel
import torch

# load base model
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "sarvamai/sarvam-1",
    max_seq_length = 512,
    dtype = torch.float16,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

def generate_from(mdl, tok, prompt, max_new_tokens=256):
    text = (
        f"### System:\nतुम एक सहायक हिन्दी भाषा के सहायक हो।\n\n"
        f"### User:\n{prompt}\n\n"
        f"### Assistant:\n"
    )
    inputs = tok(
        text,
        return_tensors = "pt",
        truncation = True,
        max_length = 512
    ).to("cuda")

    outputs = mdl.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = max_new_tokens,
        temperature = 0.3,
        top_p = 0.85,
        top_k = 50,
        repetition_penalty = 1.15,
        do_sample = True,
        pad_token_id = tok.eos_token_id
    )
    return tok.decode(
        outputs[0][len(inputs["input_ids"][0]):],
        skip_special_tokens = True
    ).strip()

# 5 test prompts covering all domains
test_prompts = [
    "महात्मा गांधी के बारे में बताइए।",
    "भारत का सबसे बड़ा रेगिस्तान कौन सा है?",
    "सुबह जल्दी उठने की आदत कैसे डालें?",
    "एक किसान के पास 120 आम हैं। वह उन्हें 8 टोकरियों में बराबर बाँटता है। हर टोकरी में कितने आम होंगे?",
    "दोस्त से झगड़ा हो जाए तो क्या करें?",
]

# store responses for Cell 10
base_responses = []
ft_responses = []

print("=" * 70)
print("         BASE MODEL vs FINE-TUNED MODEL COMPARISON")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    print(f"\nप्रश्न {i+1}: {prompt}")
    print("-" * 70)

    base_resp = generate_from(base_model, base_tokenizer, prompt)
    ft_resp   = generate(prompt)

    base_responses.append(base_resp)
    ft_responses.append(ft_resp)

    print(f"❌ BASE MODEL:\n{base_resp}")
    print(f"\n✅ FINE-TUNED MODEL:\n{ft_resp}")
    print("=" * 70)

print("\nComparison complete. Run Cell 10 for metrics.")

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

sarvamai/sarvam-1 does not have a padding token! Will use pad_token = <unk>.
         BASE MODEL vs FINE-TUNED MODEL COMPARISON

प्रश्न 1: महात्मा गांधी के बारे में बताइए।
----------------------------------------------------------------------
❌ BASE MODEL:
Gandhiji, Mohammed Ali Jinnah और Bhagat Singh जैसे कई लोगों को प्रेरित करते थे. उन्होंने 1947 में भारत की स्वतंत्रता प्राप्त करने के लिए अहिंसक प्रतिरोध का उपयोग किया था. उनके नेतृत्व ने दुनिया भर में नागरिक अधिकारों और स्वतंत्रता आंदोलनों पर गहरा प्रभाव डाला है. </s>

✅ FINE-TUNED MODEL:
महात्मा गांधी भारत की स्वतंत्रता आंदोलन के प्रमुख नेता थे जिन्होंने अहिंसा और सत्याग्रह का उपयोग किया। उन्होंने अंग्रेजों के खिलाफ शांतिपूर्ण विरोध प्रदर्शन किए जिससे देश को आजादी मिली। उनका जीवन सत्य, प्रेम और सेवा पर आधारित था जो आज भी प्रेरणादायक है। वे सभी धर्मों के प्रति सहिष्णुता और समानता में विश्वास रखते थे। उनकी शिक्षाएँ विश्व स्तर पर प्रभावशाली हैं और भारतीय संस्कृति का महत्वपूर्ण हिस्सा बन गई हैं। महात्मा गांधी ने राष्ट्रपिता की उपाधि प्र

In [11]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
import numpy as np
nltk.download("punkt", quiet=True)

smoothing = SmoothingFunction().method1
base_bleu_scores = []
ft_bleu_scores   = []
base_lengths     = []
ft_lengths       = []

print("Calculating metrics on 30 held-out test examples...")
print("Please wait...\n")

for example in test_dataset.select(range(min(30, len(test_dataset)))):
    text = example["text"]
    parts = text.split("### Assistant:\n")
    if len(parts) < 2:
        continue

    reference = parts[1].strip()
    user_part = parts[0].split("### User:\n")[-1].split("\n\n")[0].strip()

    base_hyp = generate_from(base_model, base_tokenizer, user_part)
    ft_hyp   = generate(user_part)

    ref_tokens  = list(reference)
    base_tokens = list(base_hyp)
    ft_tokens   = list(ft_hyp)

    if base_tokens:
        base_bleu_scores.append(
            sentence_bleu([ref_tokens], base_tokens, smoothing_function=smoothing)
        )
        base_lengths.append(len(base_hyp.split()))

    if ft_tokens:
        ft_bleu_scores.append(
            sentence_bleu([ref_tokens], ft_tokens, smoothing_function=smoothing)
        )
        ft_lengths.append(len(ft_hyp.split()))

# compute metrics
avg_base_bleu  = np.mean(base_bleu_scores) if base_bleu_scores else 0
avg_ft_bleu    = np.mean(ft_bleu_scores)   if ft_bleu_scores   else 0
bleu_improve   = ((avg_ft_bleu - avg_base_bleu) / avg_base_bleu * 100) if avg_base_bleu > 0 else 0

avg_base_len   = np.mean(base_lengths) if base_lengths else 0
avg_ft_len     = np.mean(ft_lengths)   if ft_lengths   else 0

base_empty     = sum(1 for r in base_responses if len(r.strip()) < 10)
ft_empty       = sum(1 for r in ft_responses   if len(r.strip()) < 10)

print("=" * 70)
print("              QUANTITATIVE EVALUATION RESULTS")
print("=" * 70)
print(f"\n{'Metric':<35} {'Base Model':>15} {'Fine-tuned':>15}")
print("-" * 70)
print(f"{'BLEU Score (avg)':<35} {avg_base_bleu:>15.4f} {avg_ft_bleu:>15.4f}")
print(f"{'BLEU Improvement':<35} {'—':>15} {f'+{bleu_improve:.1f}%':>15}")
print(f"{'Avg response length (words)':<35} {avg_base_len:>15.1f} {avg_ft_len:>15.1f}")
print(f"{'Empty/incoherent responses':<35} {base_empty:>15} {ft_empty:>15}")
print(f"{'Test examples evaluated':<35} {len(base_bleu_scores):>15} {len(ft_bleu_scores):>15}")
print("=" * 70)

print(f"""
INTERPRETATION:
- BLEU score measures how similar generated text is to reference answers
- Higher BLEU = closer to expected Hindi responses
- Fine-tuned model improved by {bleu_improve:.1f}% over base model
- Longer responses indicate the model learned to give detailed answers
- Fewer empty responses means more reliable instruction following
""")

# per domain breakdown
domains = [
    "Indian culture & history",
    "General knowledge",
    "Everyday instructions",
    "Math reasoning",
    "Polite conversation"
]

print("PER-PROMPT COMPARISON (from Cell 9 prompts):")
print("-" * 70)
for i, (domain, base_r, ft_r) in enumerate(zip(domains, base_responses, ft_responses)):
    base_words = len(base_r.split())
    ft_words   = len(ft_r.split())
    print(f"{i+1}. {domain}")
    print(f"   Base: {base_words} words | Fine-tuned: {ft_words} words")
print("=" * 70)

Calculating metrics on 30 held-out test examples...
Please wait...

              QUANTITATIVE EVALUATION RESULTS

Metric                                   Base Model      Fine-tuned
----------------------------------------------------------------------
BLEU Score (avg)                             0.2042          0.2118
BLEU Improvement                                  —           +3.7%
Avg response length (words)                   115.6           159.4
Empty/incoherent responses                        0               0
Test examples evaluated                          30              30

INTERPRETATION:
- BLEU score measures how similar generated text is to reference answers
- Higher BLEU = closer to expected Hindi responses
- Fine-tuned model improved by 3.7% over base model
- Longer responses indicate the model learned to give detailed answers
- Fewer empty responses means more reliable instruction following

PER-PROMPT COMPARISON (from Cell 9 prompts):
------------------------------